[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pdiprodi/mjlab/blob/feature/motor-database-extension/notebooks/humanoid_motor_demo_easy.ipynb)

# Unitree G1 Humanoid - Automatic Motor/Battery Demo

This notebook demonstrates the **new automatic API** for motor and battery physics.

## ✨ What's New?

With the automatic API, you get:
- ✅ **Auto-discovery** of motors from XML
- ✅ **Auto-discovery** of battery from XML
- ✅ **Automatic** motor current calculation
- ✅ **Automatic** battery updates (SOC, voltage, temperature)
- ✅ **Zero manual physics** - just load and step!

## 📚 What You'll Learn:
1. 🤖 Load a robot with embedded motor/battery specs
2. 🎮 Send position commands to joints
3. 📊 Automatically track all battery metrics
4. 📈 Visualize battery dynamics

**90% less code than manual approach!**

In [ ]:
# Check if running in Google Colab
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
  print("Running in Google Colab - installing mjlab...")
  !pip install -q git+https://github.com/pdiprodi/mjlab.git@feature/motor-database-extension

  # Download the pre-configured model
  print("\nDownloading pre-configured G1 model with motors and battery...")
  !mkdir -p model/motors model/batteries
  !wget -q https://raw.githubusercontent.com/pdiprodi/mjlab/feature/motor-database-extension/notebooks/model/g1_with_motors_battery.xml -O model/g1_with_motors_battery.xml
  !wget -q https://raw.githubusercontent.com/pdiprodi/mjlab/feature/motor-database-extension/notebooks/model/motors/unitree_7520_14.json -O model/motors/unitree_7520_14.json
  !wget -q https://raw.githubusercontent.com/pdiprodi/mjlab/feature/motor-database-extension/notebooks/model/motors/unitree_5020_9.json -O model/motors/unitree_5020_9.json
  !wget -q https://raw.githubusercontent.com/pdiprodi/mjlab/feature/motor-database-extension/notebooks/model/batteries/unitree_g1_9ah.json -O model/batteries/unitree_g1_9ah.json

  print("✓ Installation complete!")
else:
  print("Running locally - assuming mjlab is already installed")

In [ ]:
import mujoco
import torch
import numpy as np
import matplotlib.pyplot as plt

from mjlab.scene import Scene, SceneCfg
from mjlab.entity import EntityCfg
from mjlab.sim import Simulation, SimulationCfg

print("✓ All imports successful!")

## Step 1: Load Robot with Automatic Discovery

The new API automatically:
- Discovers motor specs from XML `<custom><text>` elements
- Discovers battery specs from XML
- Creates ElectricalMotorActuators
- Creates BatteryManager

**Just 3 lines of code!**

In [ ]:
# Create scene with automatic motor/battery discovery
scene_cfg = SceneCfg(
  num_envs=1,
  entities={
    "g1": EntityCfg(
      spec_fn=lambda: mujoco.MjSpec.from_file("model/g1_with_motors_battery.xml"),
      # auto_discover_motors=True,  # Default - motors auto-discovered!
    )
  },
  # auto_battery=True,  # Default - battery auto-discovered!
)

scene = Scene(scene_cfg, device="cpu")

print("✓ Scene created with automatic discovery!")
print(f"  Entities: {list(scene.entities.keys())}")
print(f"  Battery: {scene._battery_manager is not None}")

In [ ]:
# Compile and initialize
mj_model = scene.compile()
sim = Simulation(num_envs=1, cfg=SimulationCfg(), model=mj_model, device="cpu")
scene.initialize(mj_model, sim.model, sim.data)

# Get references
battery = scene._battery_manager
g1 = scene.entities["g1"]

print("✓ Simulation initialized")
print(f"\nRobot Details:")
print(f"  Bodies: {mj_model.nbody}")
print(f"  Joints: {mj_model.njnt}")
print(f"  Actuators: {mj_model.nu}")
print(f"  Auto-created motor actuators: {len(g1._actuators)}")

print(f"\nBattery Initial State:")
print(f"  SOC: {battery.soc[0].item() * 100:.1f}%")
print(f"  Voltage: {battery.voltage[0].item():.2f}V")
print(f"  Temperature: {battery.temperature[0].item():.1f}°C")
print(
  f"  Capacity: {battery.cfg.battery_spec.capacity_ah:.1f}Ah ({battery.cfg.battery_spec.energy_wh:.1f}Wh)"
)

## Step 2: Find Joint Names

Let's find the arm joint names so we can control them.

In [ ]:
# Get all joint names
all_joints = g1.joint_names

print(f"Total joints: {len(all_joints)}")
print(f"\nArm joints:")
arm_joints = [j for j in all_joints if "shoulder" in j or "elbow" in j]
for i, joint in enumerate(arm_joints, 1):
  print(f"  {i}. {joint}")

# Store arm joint indices for later use
arm_joint_indices = [all_joints.index(j) for j in arm_joints]

## Step 3: Simulate Arm Movements

We'll send sinusoidal position targets to arm joints.

**The magic:**
- Motors automatically calculate current from torque
- Battery automatically aggregates currents
- Battery automatically updates SOC/voltage/temperature
- All during `scene.write_data_to_sim()` and `scene.update(dt)`!

In [ ]:
# Simulation parameters
dt = sim.mj_model.opt.timestep  # MuJoCo timestep (typically 0.002s)
num_steps = 2500  # ~5 seconds at 500Hz
movement_frequency = 0.5  # 0.5 Hz arm movements

# Storage for visualization
time_history = []
soc_history = []
voltage_history = []
current_history = []
temp_history = []
power_history = []

print(f"Simulating {num_steps} steps ({num_steps * dt:.1f}s) with automatic physics...")
print(f"  Movement frequency: {movement_frequency} Hz")
print(f"  Timestep: {dt:.4f}s ({1 / dt:.0f}Hz)")
print(f"\n⚡ Motor currents and battery state calculated AUTOMATICALLY!\n")

for step in range(num_steps):
  time = step * dt

  # Generate sinusoidal position targets
  shoulder_pitch = 0.8 * np.sin(2 * np.pi * movement_frequency * time)
  shoulder_roll = 0.3 * abs(np.sin(2 * np.pi * movement_frequency * time))
  elbow = 0.8 * (1 + np.sin(2 * np.pi * movement_frequency * time)) / 2

  # Set position targets for all joints
  # (Most joints stay at zero, arms move)
  joint_targets = torch.zeros((1, len(all_joints)))

  # Set arm joint targets
  for joint_name in arm_joints:
    idx = all_joints.index(joint_name)
    if "shoulder_pitch" in joint_name:
      joint_targets[0, idx] = shoulder_pitch
    elif "shoulder_roll" in joint_name:
      joint_targets[0, idx] = shoulder_roll
    elif "elbow" in joint_name:
      joint_targets[0, idx] = elbow

  g1.set_joint_position_target(joint_targets)

  # ⚡ THE MAGIC HAPPENS HERE ⚡
  # 1. Motors compute current from torque (automatic)
  # 2. Battery aggregates currents (automatic)
  # 3. Battery voltage feedback to motors (automatic)
  scene.write_data_to_sim()

  # Step MuJoCo physics
  sim.step()

  # ⚡ MORE MAGIC ⚡
  # Battery updates SOC, temperature (automatic)
  scene.update(dt)

  # Record metrics every 5 steps
  if step % 5 == 0:
    time_history.append(time)
    soc_history.append(battery.soc[0].item())
    voltage_history.append(battery.voltage[0].item())
    current_history.append(battery.current[0].item())
    temp_history.append(battery.temperature[0].item())
    power_history.append(battery.power_out[0].item())

  # Progress indicator
  if step % 500 == 0:
    print(
      f"  Step {step}/{num_steps} ({time:.1f}s) - SOC: {battery.soc[0].item() * 100:.2f}% - Current: {battery.current[0].item():.2f}A"
    )

print(f"\n✓ Simulation complete!")
print(f"\n📊 Final State (all calculated automatically):")
print(f"  Battery SOC: {battery.soc[0].item() * 100:.2f}%")
print(f"  Battery voltage: {battery.voltage[0].item():.2f}V")
print(f"  Current draw: {battery.current[0].item():.2f}A")
print(f"  Temperature: {battery.temperature[0].item():.2f}°C")
print(f"  Power: {battery.power_out[0].item():.1f}W")

print(f"\n📈 Current Statistics:")
print(f"  Average: {np.mean(current_history):.2f}A")
print(f"  Peak: {np.max(current_history):.2f}A")
print(f"  Min: {np.min(current_history):.2f}A")

## Step 4: Visualize Battery Dynamics

All these metrics were calculated automatically!

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
  "Unitree G1 Battery Dynamics - Automatic Physics Calculation",
  fontsize=16,
  fontweight="bold",
)

# SOC over time
axes[0, 0].plot(time_history, [s * 100 for s in soc_history], "b-", linewidth=2.5)
axes[0, 0].set_xlabel("Time (s)", fontsize=11)
axes[0, 0].set_ylabel("State of Charge (%)", fontsize=11)
axes[0, 0].set_title("Battery SOC (Auto-Updated)", fontsize=12, fontweight="bold")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim([min(s * 100 for s in soc_history) - 0.1, 100.1])

# Voltage over time
axes[0, 1].plot(time_history, voltage_history, "g-", linewidth=2.5)
axes[0, 1].set_xlabel("Time (s)", fontsize=11)
axes[0, 1].set_ylabel("Voltage (V)", fontsize=11)
axes[0, 1].set_title(
  "Battery Terminal Voltage (Auto-Feedback to Motors)", fontsize=12, fontweight="bold"
)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(
  y=battery.cfg.battery_spec.nominal_voltage,
  color="k",
  linestyle="--",
  alpha=0.5,
  label="Nominal",
)
axes[0, 1].legend(fontsize=10)

# Current over time
axes[1, 0].plot(time_history, current_history, "r-", linewidth=2.5)
axes[1, 0].set_xlabel("Time (s)", fontsize=11)
axes[1, 0].set_ylabel("Current (A)", fontsize=11)
axes[1, 0].set_title(
  "Battery Current (Auto-Calculated from Motor Torques)",
  fontsize=12,
  fontweight="bold",
)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axhline(
  y=np.mean(current_history),
  color="orange",
  linestyle="--",
  alpha=0.7,
  linewidth=2,
  label=f"Avg: {np.mean(current_history):.2f}A",
)
axes[1, 0].legend(fontsize=10)

# Temperature over time
axes[1, 1].plot(time_history, temp_history, "purple", linewidth=2.5)
axes[1, 1].set_xlabel("Time (s)", fontsize=11)
axes[1, 1].set_ylabel("Temperature (°C)", fontsize=11)
axes[1, 1].set_title(
  "Battery Temperature (Auto-Thermal Model)", fontsize=12, fontweight="bold"
)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(
  y=battery.cfg.battery_spec.ambient_temperature,
  color="k",
  linestyle="--",
  alpha=0.5,
  label="Ambient",
)
axes[1, 1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("✓ Visualization complete!")
print("\n⚡ All metrics calculated automatically - zero manual physics!")

## Summary

### ✅ What We Did

1. **Loaded** robot with automatic motor/battery discovery (3 lines)
2. **Set** joint position targets
3. **Stepped** simulation with `write_data_to_sim()` and `update()`
4. **Got** automatic calculations:
   - ⚡ Motor currents from torque
   - ⚡ Battery current aggregation
   - ⚡ Voltage sag from internal resistance
   - ⚡ SOC discharge tracking
   - ⚡ Thermal heating (I²R)
   - ⚡ Voltage feedback to motors

### 🎯 Key Benefits vs Manual Approach

| Feature | Manual (Old) | Automatic (New) |
|---------|-------------|----------------|
| **Code Lines** | ~200 lines | **~20 lines** |
| **Motor Discovery** | Manual config | **Automatic from XML** |
| **Battery Discovery** | Manual config | **Automatic from XML** |
| **Current Calculation** | Manual equation | **Automatic** |
| **Battery Update** | Manual calls | **Automatic** |
| **Physics Knowledge** | Required | **Not needed** |

### 🔑 Key Observations

- **Current** automatically calculated from PD control torques
- **Voltage** sags under load (internal resistance R)
- **SOC** decreases linearly with current draw
- **Temperature** increases from I²R heating in battery
- **Voltage feedback** limits motor performance when battery drains

### 🚀 Next Steps

1. **Try different movements**: Control legs for walking simulation
2. **Optimize for energy**: Find movements that minimize current
3. **Mission planning**: Predict battery runtime for tasks
4. **Train RL policies**: Use this setup with reinforcement learning

### 📚 Resources

- [mjlab Repository](https://github.com/pdiprodi/mjlab/tree/feature/motor-database-extension)
- [Design Proposal](https://github.com/pdiprodi/mjlab/blob/feature/motor-database-extension/docs/motors/design-proposal.md)
- [Full Demo](humanoid_motor_demo.ipynb) - How to create models with specs

---

**🎉 90% less code with automatic motor/battery physics!**

Created with [mjlab](https://github.com/pdiprodi/mjlab)